# Day 68：面试专题(二)——项目深挖与系统设计：应对'项目被问穿'与十万 DAU 企业知识库架构（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 10 周**：毕业答辩、简历作品集与面试冲刺结业  
> **GitLab 工单**：`STARMIND-068` ｜ **开发分支**：`feat/day68-system-design-capacity`  
> **配套 HTML5 课件**：`courseware/day68/index.html` ｜ **独立版本快照**：`versions/day68/`

---

## 🎙️ 老陈开场旁白

早啊各位！很多同学一面手撕全过，却倒在二面的两句话上：第一句是面试官盯着你的星辰智脑项目连追 5 层——'你为什么要做 RRF 不直接加权？权重怎么调？数据量涨到 500 万篇文档怎么办？索引更新延迟怎么控？' 也就是俗称的'项目被问穿'；第二句是系统设计题——'请你从零设计一个日活 10 万（100K DAU）的企业知识库 + Agent 平台，算一下峰值 QPS、向量库内存、需要几张 GPU 卡、一个月花多少钱？' 今天咱们把这两道大题彻底拆透，直接写出一个十万 DAU 容量估算引擎和 5 层防问穿 ADR 架构决策生成器！

### 承上启下
- **昨日基线**：昨天我们手撕了一面高频算法与后端工程题，今天攻克决定职级与薪资上限的二面/三面重头戏：项目深挖与高并发系统设计！
- **明日预告**：明天（Day 69）是全真模拟面试日：技术面 + HR 面全流程打分评估与个性化 7 天查漏补缺改进清单。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day68'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-068）

高级 AI 工程师二/三面核心考察两点：一是候选人对自研项目每个架构决策的深度权衡（防问穿）；二是面对十万级 DAU 大规模企业知识库时的容量估算（Back-of-the-Envelope Estimation）与高可用架构设计能力。

**今日迭代目标：**
- 实现十万级 DAU 企业 RAG + Agent 平台容量与成本估算引擎（峰值 QPS、TPM、向量库 RAM、GPU 卡数与云/自建成本对比）。
- 实现'防问穿' 5 层项目深挖演练与架构决策记录（ADR: Context/Options/Decision/Tradeoffs/Scale-out）校验器。

**验收标准：**
- 容量估算器能根据 DAU、人均日提问数、峰值系数、文档切片数准确推导出 Peak QPS、向量内存(GB) 与所需 vLLM GPU 数量。
- ADR 防问穿校验器确保每个核心架构决策均具备备选方案对比、量化依据与 10 倍扩容演进路径。
- 纯 Python 离线运行，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 系统设计面试黄金 4 步节奏（45 分钟）

1) 澄清需求与边界（5m）：确认 DAU、文档规模、是否多租户、SLA 延迟要求；2) 容量估算（5m）：算出 QPS、存储、显存与成本量级；3) 高层架构与核心数据流（15m）：画清读写两条主链路；4) 深挖瓶颈与容灾（20m）：主动聊热点缓存雪崩、RAG 权限隔离、模型超时降级与灰度发布。

> **⚠️ 老陈避坑**：切忌不问需求上来就画一堆微服务框图；先对齐量化约束条件是区分中级与高级工程师的分水岭。

### 2. 怎么回答'1000 万文档更新时如何避免检索延迟和脏读'？

采用异步增量索引（文档变更发 Kafka -> Worker 仅重算变更段落哈希 -> 幂等 Upsert 写入向量库与 ES），并对大版本全量重建采用'双索引别名无缝切换（Blue/Green Index Alias）'，切换瞬间零停机。

> **⚠️ 老陈避坑**：不要在用户上传接口里同步做 PDF OCR + Embedding，大文件动辄 30 秒直接把 HTTP 网关打超时。


## 💻 实操步骤 1：步骤一：实现十万 DAU 企业知识库容量、向量内存与 GPU 算力估算引擎

- **目标文件**：`versions/day68/src/day68_capacity_estimator.py`
- **老陈代码旁白**：将系统设计面试中的核心容量估算公式代码化，一键推导出 Peak QPS、向量库内存、vLLM GPU 副本数与月度费用对比。


In [ ]:
"""
Day 68 - 模块一：十万 DAU 企业级 RAG + Agent 平台系统设计容量与成本估算引擎
"""
from __future__ import annotations

import math
from dataclasses import dataclass


@dataclass
class ScaleSpec:
    dau: int = 100_000
    queries_per_user_day: float = 10.0
    peak_factor: float = 5.0
    cache_hit_rate: float = 0.35
    avg_tokens_per_query: int = 2000
    total_chunks: int = 10_000_000
    embedding_dim: int = 1024
    use_sq8_quant: bool = True
    single_gpu_tps: float = 1500.0  # 单张 A100/L40S 跑 7B vLLM 吞吐估算(tokens/s)
    local_route_ratio: float = 0.70


@dataclass
class CapacityReport:
    daily_requests: int
    avg_qps: float
    peak_qps: float
    llm_peak_qps: float
    vector_ram_gb: float
    required_gpus: int
    monthly_cloud_only_cny: float
    monthly_hybrid_cny: float


def estimate_capacity(spec: ScaleSpec) -> CapacityReport:
    daily_reqs = int(spec.dau * spec.queries_per_user_day)
    avg_qps = daily_reqs / 86400.0
    peak_qps = avg_qps * spec.peak_factor
    llm_peak_qps = peak_qps * (1.0 - spec.cache_hit_rate)

    # 向量内存估算：FP32 为 4 bytes/dim，SQ8 标量量化为 1 byte/dim，HNSW 图索引开销约 1.5x
    bytes_per_dim = 1 if spec.use_sq8_quant else 4
    raw_bytes = spec.total_chunks * spec.embedding_dim * bytes_per_dim * 1.5
    vector_ram_gb = round(raw_bytes / (1024 ** 3), 2)

    # 本地 7B 峰值输出吞吐需求（按总 token 的 30% 为输出 decode token 估算 + 30% 冗余）
    local_peak_decode_tps = llm_peak_qps * spec.local_route_ratio * (spec.avg_tokens_per_query * 0.30)
    required_gpus = max(1, math.ceil((local_peak_decode_tps * 1.3) / max(1.0, spec.single_gpu_tps)))

    # 月度成本对比（云旗舰均价 0.02 元/千Token；单 GPU 月租 6000 元）
    monthly_tokens_k = (daily_reqs * 30 * spec.avg_tokens_per_query) / 1000.0
    cloud_only_cny = round(monthly_tokens_k * 0.02, 2)
    hybrid_cloud_part = monthly_tokens_k * (1.0 - spec.cache_hit_rate) * (1.0 - spec.local_route_ratio) * 0.02
    hybrid_cny = round(hybrid_cloud_part + required_gpus * 6000.0, 2)

    return CapacityReport(
        daily_requests=daily_reqs,
        avg_qps=round(avg_qps, 2),
        peak_qps=round(peak_qps, 2),
        llm_peak_qps=round(llm_peak_qps, 2),
        vector_ram_gb=vector_ram_gb,
        required_gpus=required_gpus,
        monthly_cloud_only_cny=cloud_only_cny,
        monthly_hybrid_cny=hybrid_cny,
    )


if __name__ == "__main__":
    rep = estimate_capacity(ScaleSpec())
    print("十万 DAU 容量估算报告:", rep)


### 🔍 步骤 1 代码深度解析

清晰展示系统设计中每个数字的来龙去脉：DAU -> Peak QPS -> 缓存削峰 -> 向量量化省 75% 内存 -> 算清自建 GPU 与纯云 API 的月度总账。

**预期运行输出参考：**
```text
十万 DAU 容量估算报告: CapacityReport(daily_requests=1000000, avg_qps=11.57, peak_qps=57.87, llm_peak_qps=37.62, vector_ram_gb=14.31, required_gpus=..., ...)
```


## 💻 实操步骤 2：步骤二：实现'防问穿' 5 层项目深挖架构决策记录（ADR）校验器

- **目标文件**：`versions/day68/src/day68_deep_dive_defense.py`
- **老陈代码旁白**：把星辰智脑每个关键技术选型沉淀为 5 层 ADR 卡片，面试前逐张自检是否具备备选对比、真实踩坑、量化实证与 10x 扩容方案。


In [ ]:
"""
Day 68 - 模块二：应对'项目被问穿'的 5 层架构决策记录 (ADR) 生成与完整度校验器
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class ArchitectureDecisionCard:
    topic: str
    l1_problem: str
    l2_alternatives: List[str]
    l3_pitfall_lesson: str
    l4_metric_proof: str
    l5_scale_10x_plan: str

    def to_markdown(self) -> str:
        alts = " / ".join(self.l2_alternatives)
        return (
            f"### ADR: {self.topic}\n"
            f"- **L1 痛点与目标**: {self.l1_problem}\n"
            f"- **L2 备选方案权衡**: {alts}\n"
            f"- **L3 真实踩坑与解法**: {self.l3_pitfall_lesson}\n"
            f"- **L4 量化指标实证**: {self.l4_metric_proof}\n"
            f"- **L5 十倍扩容演进**: {self.l5_scale_10x_plan}\n"
        )


def audit_adr_readiness(card: ArchitectureDecisionCard) -> Dict[str, object]:
    missing: List[str] = []
    if len(card.l1_problem.strip()) < 8:
        missing.append("L1 痛点描述过于空泛")
    if len(card.l2_alternatives) < 2:
        missing.append("L2 至少需对比 2 种备选技术方案及其取舍")
    if len(card.l3_pitfall_lesson.strip()) < 8:
        missing.append("L3 缺少真实踩坑排障细节，容易被面试官怀疑是纸上谈兵")
    if not any(ch.isdigit() for ch in card.l4_metric_proof):
        missing.append("L4 缺少量化指标数字支撑")
    if len(card.l5_scale_10x_plan.strip()) < 8:
        missing.append("L5 缺少流量或数据量扩大 10 倍后的演进预案")
    return {"ready": len(missing) == 0, "missing": missing}


if __name__ == "__main__":
    rag_adr = ArchitectureDecisionCard(
        topic="为什么采用 BM25 + 向量 + RRF 而不是单一向量检索？",
        l1_problem="企业制度包含大量专有错误码与编号，纯稠密向量召回率仅 62%。",
        l2_alternatives=["纯向量检索(同义强但专词差)", "线性加权融合(量纲难调参)", "RRF倒数秩融合(无量纲免调参)"],
        l3_pitfall_lesson="初期英文错误码被中文分词拆散，改为正则保留完整字母数字 token + 中文 2-gram 后精准命中。",
        l4_metric_proof="黄金集 Context Recall 由 62% 提升至 91%，检索 P95 保持在 35ms 内。",
        l5_scale_10x_plan="千万切片规模下切换 Milvus(SQ8量化) + ES 分布式分片，前置 RBAC 部门分区过滤。",
    )
    print("ADR 备战体检:", audit_adr_readiness(rag_adr))
    print(rag_adr.to_markdown())


### 🔍 步骤 2 代码深度解析

强迫自己把每个亮点技术都补齐 L1~L5 五层纵深，二面再凶的面试官顺着往下挖，挖到的全是你提前布好的得分点！

**预期运行输出参考：**
```text
ADR 备战体检: {'ready': True, 'missing': []}
### ADR: 为什么采用 BM25 + 向量 + RRF 而不是单一向量检索？...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day68/tests/test_day68.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 68 单元测试：十万 DAU 容量估算引擎与 5 层 ADR 防问穿校验"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day68_capacity_estimator import ScaleSpec, estimate_capacity
from src.day68_deep_dive_defense import ArchitectureDecisionCard, audit_adr_readiness


def test_capacity_estimator_100k_dau_and_quantization():
    fp32_rep = estimate_capacity(ScaleSpec(use_sq8_quant=False))
    sq8_rep = estimate_capacity(ScaleSpec(use_sq8_quant=True))
    assert fp32_rep.daily_requests == 1_000_000
    assert 55.0 <= fp32_rep.peak_qps <= 60.0
    assert sq8_rep.vector_ram_gb < fp32_rep.vector_ram_gb * 0.3
    assert sq8_rep.monthly_hybrid_cny < sq8_rep.monthly_cloud_only_cny


def test_adr_audit_passes_complete_and_catches_shallow_card():
    good = ArchitectureDecisionCard(
        topic="HITL 审批门禁设计",
        l1_problem="高额报销自动提交存在资损风险。",
        l2_alternatives=["纯Prompt约束(易越狱)", "代码层确定性策略+审批状态机(强可控)"],
        l3_pitfall_lesson="前端双击导致重复审批打款，增加 PENDING 终态幂等锁返回 409。",
        l4_metric_proof="高危误执行率降至 0%，审批接口 P95 耗时 25ms。",
        l5_scale_10x_plan="审批状态迁移至 PostgreSQL 行级悲观锁 + 飞书审批卡片异步回调。",
    )
    shallow = ArchitectureDecisionCard("选型", "需要做", ["只看了一个"], "没踩坑", "效果不错", "加机器")
    assert audit_adr_readiness(good)["ready"] is True
    assert audit_adr_readiness(shallow)["ready"] is False
    assert len(audit_adr_readiness(shallow)["missing"]) >= 4

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：比较 1 万 / 10 万 / 50 万 DAU 三个规模下的 GPU 需求曲线

**题目要求：**
循环调用 `estimate_capacity(ScaleSpec(dau=d))` 输出三个规模下的 `peak_qps` 与 `required_gpus`。

**思路提示：** 遍历 `[10_000, 100_000, 500_000]` 构造 `ScaleSpec` 即可。


In [ ]:
from src.day68_capacity_estimator import ScaleSpec, estimate_capacity

for d in (10_000, 100_000, 500_000):
    r = estimate_capacity(ScaleSpec(dau=d))
    print(f"DAU={d:>7} -> Peak QPS={r.peak_qps:>6}, GPUs={r.required_gpus}, 月混合成本={r.monthly_hybrid_cny}")


**答案解析与评分标准：**

理解不同量级下的资源拐点，有助于在面试中针对创业公司（1万 DAU）和大厂（50万 DAU）给出贴合实际的架构方案。


### 作业 2（企业实战）：企业实战：计算多租户向量分片数量建议

**题目要求：**
假设单个向量分片（Shard）最佳实践承载不超过 200 万切片，编写 `recommend_shards(total_chunks, max_per_shard=2_000_000)` 返回推荐分片数。

**思路提示：** 使用 `max(1, math.ceil(total_chunks / max_per_shard))`。


In [ ]:
import math


def recommend_shards(total_chunks: int, max_per_shard: int = 2_000_000) -> int:
    return max(1, math.ceil(total_chunks / max_per_shard))


assert recommend_shards(10_000_000) == 5
assert recommend_shards(500_000) == 1
print("向量分片数量规划通过")


**答案解析与评分标准：**

合理分片能让单次检索并发扫描并控制单分片 HNSW 构建内存。


### 作业 3（架构进阶）：架构进阶：评估系统设计的单点故障（SPOF）风险检测

**题目要求：**
给定各组件部署副本数字典（如 `{'nginx': 2, 'api': 4, 'redis': 1}`），编写 `find_spof(replicas)` 找出所有副本数 `< 2` 的单点故障组件。

**思路提示：** 过滤 `v < 2` 的键名列表。


In [ ]:
def find_spof(replicas: dict[str, int]) -> list[str]:
    return [k for k, v in replicas.items() if v < 2]


spofs = find_spof({"nginx": 2, "api": 4, "redis": 1, "postgres": 1})
assert spofs == ["redis", "postgres"]
print("高可用 SPOF 风险项检测通过:", spofs)


**答案解析与评分标准：**

高可用系统设计收尾时主动排查一遍全链路单点故障（SPOF），是三面架构加分利器。


---

## 🎯 老陈收尾复盘

硬核！以后面试官只要出'设计十万 DAU 企业 RAG/Agent 平台'，你三分钟内就能在白板上把 Peak QPS、TPM、HNSW 内存、vLLM GPU 卡数和月度云端/私有化成本算得明明白白，再配合 5 层 ADR 权衡分析，直接拉出高级工程师的气场！
